In [3]:
from __future__ import annotations
import serial
import time
from typing import Optional

In [4]:
### Find the correct COM port
### "USB to UART Bridge" is for 485
### for CAN

from serial.tools import list_ports

for port in list_ports.comports():
    print(
        f"{port.device:8} "
        f"{port.description} "
        f"VID={port.vid} PID={port.pid}"
    )

COM7     Silicon Labs CP210x USB to UART Bridge (COM7) VID=4292 PID=60000
COM6     USB Serial Device (COM6) VID=5840 PID=4478
COM3     Intel(R) Active Management Technology - SOL (COM3) VID=None PID=None


Attention: Close any GUI program before running this cell! Only one program can normally own a COM port at a time.

In [5]:
PORT = 'COM6'   # Change this to CAN accordingly
USB_BAUDRATE = 1_000_000 # Sets the serial communication speed between the computer and USB adapter
READ_TIMEOUT = 0.10      # seconds

print('Selected port:', PORT)

ser = serial.Serial(
    port=PORT,
    baudrate=USB_BAUDRATE,
    timeout=READ_TIMEOUT,
    write_timeout=1.0,
    parity=serial.PARITY_NONE,
    bytesize=serial.EIGHTBITS, # by default
    stopbits=serial.STOPBITS_ONE,
)

print('Port open:', ser.is_open)
print('Port settings:', ser.get_settings())

Selected port: COM6
Port open: True
Port settings: {'baudrate': 1000000, 'bytesize': 8, 'parity': 'N', 'stopbits': 1, 'xonxoff': False, 'dsrdtr': False, 'rtscts': False, 'timeout': 0.1, 'write_timeout': 1.0, 'inter_byte_timeout': None}


In [6]:
def send_raw(command: bytes, pause: float = 0.05) -> None:
    """Send raw bytes to the adapter and print exactly what was sent."""
    if not ser.is_open:
        raise RuntimeError('Serial port is closed.')
    ser.write(command)
    ser.flush()
    print('TX raw:', repr(command))
    time.sleep(pause) #0.05 s pause


def read_one():
    """Read one carriage-return-terminated ASCII response."""
    raw = ser.read_until(b'\r')
    if not raw:
        return None
    text = raw.decode('ascii', errors='replace').strip('\r\n')
    return text


def listen(duration: float = 2.0):
    """Collect and print frames received during a fixed time interval."""
    frames = []
    deadline = time.monotonic() + duration

    while time.monotonic() < deadline:
        frame = read_one(timeout=0.05)
        if frame:
            frames.append(frame)
            print('RX:', frame)

    if not frames:
        print('No complete CR-terminated frame received.')

    return frames

In [7]:
ser.reset_input_buffer()
ser.reset_output_buffer()

send_raw(b'S8\r') #configure the CAN arbitration bitrate
send_raw(b'Y5\r') #configure the CAN-FD data bitrate
send_raw(b'O\r\n') # open the CAN channel

print('Initialization commands sent.')

TX raw: b'S8\r'
TX raw: b'Y5\r'
TX raw: b'O\r\n'
Initialization commands sent.


CANFD:
5 bytes  position
4 bytes  velocity
2 bytes  current limit
1 byte   mode
1 byte   run/stop
1 byte   Kp
1 byte   Kd
1 byte   zero command

In [8]:
def make_canfd_position(
    position_deg,
    speed_hz,
    current_a,
    kp,
    kd,
    run=False,
):
    # Position: 1 revolution = 1,048,576 counts
    position_code = round(position_deg / 360.0 * 1_048_576)

    # Velocity field uses electrical frequency in Hz
    velocity_code = round(speed_hz / 1000.0 * 8_388_608)

    # Current field: ±100 A corresponds to signed 16-bit range
    current_code = round(current_a / 100.0 * 32_768)

    if not 0 <= kp <= 255:
        raise ValueError("kp must be between 0 and 255")

    if not 0 <= kd <= 255:
        raise ValueError("kd must be between 0 and 255")

    payload = (
        position_code.to_bytes(5, byteorder="big", signed=True)
        + velocity_code.to_bytes(4, byteorder="big", signed=True)
        + current_code.to_bytes(2, byteorder="big", signed=True)
        + bytes([
            2,          # ModeSel = 2: position mode
            int(run),   # RunCmd: 0 = stopped/free, 1 = run
            kp,
            kd,
            0,          # Zero command inactive
        ])
    )

    return payload

In [9]:
#Build a safe, disabled command
#position: 0°
#speed: 1 Hz
#mode: position
#current: 0 A
#run: disabled

MOTOR_ID = 0x001

payload = make_canfd_position(
    position_deg=0.0, speed_hz=1.0,current_a=0.0,
    kp=0, kd=0, run=False,
)

print("Payload length:", len(payload))
print("Payload:", payload.hex(" ").upper())

Payload length: 16
Payload: 00 00 00 00 00 00 00 20 C5 00 00 02 00 00 00 00


In [10]:
frame = f"d{MOTOR_ID:03X}A{payload.hex().upper()}\r" #CAN-FD standard frame, 001 motor ID, A 16-byte payload

print("Serial frame:", repr(frame))

send_raw(frame.encode("ascii"))
time.sleep(0.05)
send_raw(frame.encode("ascii"))

Serial frame: 'd001A0000000000000020C500000200000000\r'
TX raw: b'd001A0000000000000020C500000200000000\r'
TX raw: b'd001A0000000000000020C500000200000000\r'


In [11]:
#tyr a small movement
#position: 10°
#speed: 1 Hz
#mode: position
#current: 0.1 A
#run: abled

KP=5
KD=5

def runtest(position):
    move_payload = make_canfd_position(
    position_deg=position,
    speed_hz=2.0,
    current_a=0.1,
    kp=KP,
    kd=KD,
    run=True,
    )



    move_frame = f"d{MOTOR_ID:03X}A{move_payload.hex().upper()}\r"

    print("Serial frame:", repr(frame))

    send_raw(move_frame.encode("ascii"))
    time.sleep(0.05)
    send_raw(move_frame.encode("ascii"))

    print("end")

    ####


move_payload = make_canfd_position(
    position_deg=10.0,
    speed_hz=1.0,
    current_a=0.1,
    kp=KP,
    kd=KD,
    run=True,
)

move_frame = f"d{MOTOR_ID:03X}A{move_payload.hex().upper()}\r"


In [10]:
move_frame = f"d{MOTOR_ID:03X}A{move_payload.hex().upper()}\r" #CAN-FD standard frame, 001 motor ID, A 16-byte payload

print("Serial frame:", repr(frame))

send_raw(move_frame.encode("ascii"))
time.sleep(0.05)
send_raw(move_frame.encode("ascii"))


time.sleep(5)
print("first test...")
runtest(180)
time.sleep(12)
print("second test...")
runtest(-180)

print("end")

Serial frame: 'd001A0000000000000020C500000200000000\r'
TX raw: b'd001A00000071C7000020C500210201050500\r'
TX raw: b'd001A00000071C7000020C500210201050500\r'
first test...
Serial frame: 'd001A0000000000000020C500000200000000\r'
TX raw: b'd001A00000800000000418900210201050500\r'
TX raw: b'd001A00000800000000418900210201050500\r'
end
second test...
Serial frame: 'd001A0000000000000020C500000200000000\r'
TX raw: b'd001AFFFFF800000000418900210201050500\r'
TX raw: b'd001AFFFFF800000000418900210201050500\r'
end
end


### Notes!
The position is absolute angle, not relative to last position. Does not simplify path for shortest travel, must traverse every angle between start and end.

Commands can be interupted by other commands if not given enough time (This is probably good, it can't get stuck), interrupt does not seem to cause any undefined behaviour.

In [11]:
# Stop and close the COM port
stop_payload = bytearray(move_payload)

# Byte 12 is RunCmd
stop_payload[12] = 0

# Build the CAN-FD serial frame
stop_frame = (
    f"d{MOTOR_ID:03X}A"
    f"{bytes(stop_payload).hex().upper()}\r"
).encode("ascii")

# Stop/disable the motor
send_raw(stop_frame)

response = read_one()
print("Stop response:", response)

# Then close the COM port
ser.close()
print("Serial port closed:", not ser.is_open)

TX raw: b'd001A00000071C7000020C500210200050500\r'
Stop response: d064AFFFFFF8E2CFFFFFFFBFFD80201002601
Serial port closed: True


In [12]:
ser.close()
print("Serial port closed:", not ser.is_open)

Serial port closed: True
